# Character Supervisor 테스트 (High-Performance Parallel)

Character Team의 고성능 병렬 실행 구조를 테스트합니다.

## 핵심 검증 내용
1. **2단계 실행 흐름**: `Identity` (Phase 1) -> `Parallel Batch` (Phase 2) -> `Aggregator` 순서 확인.
2. **병렬 실행**: Phase 2에서 6개 에이전트(Appearance, Personality, Relations, Dialogue, Stats, Inventory)가 동시에 호출되는지 확인.
3. **부분 실패 허용**: 하나의 에이전트가 실패해도 전체 프로세스가 중단되지 않는지 확인.


In [1]:
import sys, os, asyncio
import nest_asyncio
from unittest.mock import MagicMock, patch, AsyncMock

# Windows 환경 등에서 이미 실행 중인 이벤트 루프 문제 해결
nest_asyncio.apply()

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path: sys.path.insert(0, project_root)

from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, '.env'))

# State Import
from app.agents.extraction.character.state import CharacterTeamState
# Supervisor Import (to check routing logic)
from app.agents.extraction.character.supervisor import character_supervisor_node, PARALLEL_AGENTS

print(f"Project root: {project_root}")

Project root: c:\jungle\weapon\sto-link-AI-backend


## 1. Mock Data & Functions 설정

실제 LLM 호출을 피하기 위해 각 서브 에이전트의 동작을 모의(Mock)합니다.

In [2]:
# 테스트용 가짜 데이터
MOCK_IDENTITY_RESULT = {"아린": {"name": "아린", "role": "protagonist"}}
MOCK_APPEARANCE_RESULT = {"아린": {"hair": "silver"}}
MOCK_PERSONALITY_RESULT = {"아린": {"traits": ["brave"]}}
MOCK_RELATIONS_RESULT = {"아린": {"relations": []}}
MOCK_DIALOGUE_RESULT = {"아린": {"tone": "calm"}}
MOCK_STATS_RESULT = {"아린": {"strength": 10}}
MOCK_INVENTORY_RESULT = {"아린": {"items": ["sword"]}}
MOCK_AGGREGATED_RESULT = [{"name": "아린", "role": "protagonist", "stats": {"strength": 10}}]

# Mock Sub-agent Functions
async def mock_identity_node(state):
    print("🤖 [Mock] Identity Agent Running...")
    return {"char_identity": MOCK_IDENTITY_RESULT, "completed_agents": ["identity"]}

async def mock_appearance_node(state):
    print("🤖 [Mock] Appearance Agent Running...")
    return {"char_appearance": MOCK_APPEARANCE_RESULT, "completed_agents": ["appearance"]}

async def mock_personality_node(state):
    print("🤖 [Mock] Personality Agent Running...")
    return {"char_personality": MOCK_PERSONALITY_RESULT, "completed_agents": ["personality"]}

async def mock_relations_node(state):
    print("🤖 [Mock] Relations Agent Running...")
    return {"char_relations": MOCK_RELATIONS_RESULT, "completed_agents": ["relations"]}

async def mock_dialogue_node(state):
    print("🤖 [Mock] Dialogue Agent Running...")
    return {"char_dialogue_mood": MOCK_DIALOGUE_RESULT, "completed_agents": ["dialogue_mood"]}

async def mock_stats_node(state):
    print("🤖 [Mock] Stats Agent Running...")
    return {"char_stats": MOCK_STATS_RESULT, "completed_agents": ["stats"]}

async def mock_inventory_node(state):
    print("🤖 [Mock] Inventory Agent Running...")
    return {"char_inventory": MOCK_INVENTORY_RESULT, "completed_agents": ["inventory"]}

async def mock_aggregator_node(state):
    print("🔗 [Mock] Aggregator Running...")
    return {
        "extracted_characters": MOCK_AGGREGATED_RESULT,
        "completed_agents": ["aggregator"],
        "messages": [{"role": "aggregator", "content": "Aggregated successfully"}]
    }

## 2. Supervisor 라우팅 로직 단위 테스트

`character_supervisor_node` 함수가 상태에 따라 올바른 다음 단계를 반환하는지 확인합니다.
새로운 로직: Identity -> Parallel Batch -> Aggregator

In [3]:
from app.agents.extraction.character.supervisor import character_supervisor_node

async def test_supervisor_routing():
    print("🧪 Supervisor Routing Test")
    
    # Case 1: 초기 상태 -> Identity
    state_init = {"completed_agents": []}
    result_init = await character_supervisor_node(state_init)
    print(f"   [Init] Next -> {result_init['next']}")
    assert result_init['next'] == "identity", "First step MUST be identity"

    # Case 2: Identity 완료 -> Parallel Batch
    state_id_done = {"completed_agents": ["identity"]}
    result_parallel = await character_supervisor_node(state_id_done)
    print(f"   [Identity Done] Next -> {result_parallel['next']}")
    assert result_parallel['next'] == "parallel_batch", "After identity, should go to parallel batch"

    # Case 3: Parallel Batch 중 하나라도 완료(Simulated) -> Aggregator
    # Note: Parallel Batch returns ALL agents in 'completed_agents' usually.
    state_parallel_done = {"completed_agents": ["identity", "appearance", "stats"]}
    result_agg = await character_supervisor_node(state_parallel_done)
    print(f"   [Parallel Done] Next -> {result_agg['next']}")
    assert result_agg['next'] == "aggregate", "After parallel batch, should go to aggregator"

    # Case 4: Aggregator 완료 -> Done
    state_all_done = {"completed_agents": ["identity", "appearance", "stats", "aggregator"]}
    result_done = await character_supervisor_node(state_all_done)
    print(f"   [Everything Done] Next -> {result_done['next']}")
    assert result_done['next'] == "done", "Should finish after aggregation"
    
    print("✅ Routing Logic Passed")

# Run unit test
await test_supervisor_routing()

🧪 Supervisor Routing Test
   [Init] Next -> identity
   [Identity Done] Next -> parallel_batch
   [Parallel Done] Next -> aggregate
   [Everything Done] Next -> done
✅ Routing Logic Passed


## 3. 전체 그래프 실행 테스트 (End-to-End)

전체 그래프를 실행하여 병렬 실행 및 부분 실패 처리를 확인합니다.

In [4]:
from app.agents.extraction.character.state import CharacterTeamState

# 패치 경로 설정
PATCH_BASE = "app.agents.extraction.character.supervisor"

async def run_full_graph_test():
    print("\n🧪 Full Graph Execution Test (with Mocks)")
    
    # Patch all sub-agent nodes.
    with patch(f"{PATCH_BASE}.identity_extraction_node", side_effect=mock_identity_node) as m_id, \
         patch(f"{PATCH_BASE}.appearance_extraction_node", side_effect=mock_appearance_node) as m_app, \
         patch(f"{PATCH_BASE}.personality_extraction_node", side_effect=mock_personality_node) as m_pers, \
         patch(f"{PATCH_BASE}.relations_extraction_node", side_effect=mock_relations_node) as m_rel, \
         patch(f"{PATCH_BASE}.dialogue_mood_extraction_node", side_effect=mock_dialogue_node) as m_dial, \
         patch(f"{PATCH_BASE}.stats_extraction_node", side_effect=mock_stats_node) as m_stat, \
         patch(f"{PATCH_BASE}.inventory_extraction_node", side_effect=mock_inventory_node) as m_inv, \
         patch(f"{PATCH_BASE}.character_aggregator_node", side_effect=mock_aggregator_node) as m_agg:
        
        # 그래프 재생성: 패치된 함수들이 PARALLEL_AGENTS 딕셔너리에 반영되도록 함
        # 주의: PARALLEL_AGENTS는 모듈 로드 시점에 정의되므로, 여기서 patch해도 이미 임포트된 딕셔너리 값은 안 바뀔 수 있음.
        # 따라서 PARALLEL_AGENTS 자체를 조작하는 것이 안전함.
        from app.agents.extraction.character import supervisor
        
        # PARALLEL_AGENTS 값을 Mock 함수로 교체
        original_agents = supervisor.PARALLEL_AGENTS.copy()
        supervisor.PARALLEL_AGENTS["appearance"] = m_app
        supervisor.PARALLEL_AGENTS["personality"] = m_pers
        supervisor.PARALLEL_AGENTS["relations"] = m_rel
        supervisor.PARALLEL_AGENTS["dialogue_mood"] = m_dial
        supervisor.PARALLEL_AGENTS["stats"] = m_stat
        supervisor.PARALLEL_AGENTS["inventory"] = m_inv

        try:
            # 그래프 생성 및 컴파일
            from app.agents.extraction.character.supervisor import create_character_team_graph
            test_graph = create_character_team_graph().compile()
            
            # 초기 상태
            initial_state: CharacterTeamState = {
                "content": "Test content",
                "retry_count": 0,
                "completed_agents": [],
                "errors": [],
                "messages": []
            }
            
            print("🚀 Starting Graph Execution...")
            result = await test_graph.ainvoke(initial_state)
            
            # 검증
            print("\n📊 Execution Result Check:")
            
            mocks = {
                "Identity": m_id,
                "Appearance": m_app,
                "Personality": m_pers,
                "Relations": m_rel,
                "Dialogue": m_dial,
                "Stats": m_stat,
                "Inventory": m_inv,
                "Aggregator": m_agg
            }
            
            all_called = True
            for name, mock_func in mocks.items():
                if mock_func.called:
                    print(f"   ✅ {name} Agent Called")
                else:
                    print(f"   ❌ {name} Agent NOT Called")
                    all_called = False
            
            completed = result.get("completed_agents", [])
            print(f"   Completed Agents Log: {completed}")
            
            if all_called and "aggregator" in completed:
                print("\n✅ Full Graph Execution Passed")
            else:
                print("\n❌ Full Graph Execution Failed")
        finally:
            # 원상 복구
            supervisor.PARALLEL_AGENTS = original_agents

# Run full test
await run_full_graph_test()


🧪 Full Graph Execution Test (with Mocks)
🚀 Starting Graph Execution...
🤖 [Mock] Identity Agent Running...
[Character Team] Starting Parallel Phase for agents: ['appearance', 'personality', 'relations', 'dialogue_mood', 'stats', 'inventory']
🤖 [Mock] Appearance Agent Running...
🤖 [Mock] Personality Agent Running...
🤖 [Mock] Relations Agent Running...
🤖 [Mock] Dialogue Agent Running...
🤖 [Mock] Stats Agent Running...
🤖 [Mock] Inventory Agent Running...
🔗 [Mock] Aggregator Running...

📊 Execution Result Check:
   ✅ Identity Agent Called
   ✅ Appearance Agent Called
   ✅ Personality Agent Called
   ✅ Relations Agent Called
   ✅ Dialogue Agent Called
   ✅ Stats Agent Called
   ✅ Inventory Agent Called
   ✅ Aggregator Agent Called
   Completed Agents Log: ['identity', 'appearance', 'personality', 'relations', 'dialogue_mood', 'stats', 'inventory', 'aggregator']

✅ Full Graph Execution Passed
